# LSTM Model: CMAPSS FD001
Sliding-window sequence model. Run on Colab T4.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## 1. Mount Drive and load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

df = pd.read_parquet('/content/drive/MyDrive/engine-data-project/train_FD001_clean.parquet')

# uncapped RUL causes mode collapse: the model minimises MSE by predicting the mean
# capping at 125 removes the noisy high-RUL tail and gives the model a learnable target shape
df['rul'] = df['rul'].clip(upper=125)

print(df.shape)
sensor_cols = [c for c in df.columns if c.startswith('sensor_')]

## 2. Build sliding windows

In [ ]:
WINDOW_SIZE = 30

windows, labels, win_engine_ids = [], [], []

for eid, group in df.groupby('engine_id'):
    group = group.sort_values('cycle')
    sensors = group[sensor_cols].values
    ruls = group['rul'].values

    # engines shorter than WINDOW_SIZE yield no windows
    for i in range(len(group) - WINDOW_SIZE + 1):
        windows.append(sensors[i : i + WINDOW_SIZE])
        labels.append(ruls[i + WINDOW_SIZE - 1])
        win_engine_ids.append(eid)

X = np.array(windows, dtype=np.float32)       # (n_windows, 30, n_sensors)
y = np.array(labels, dtype=np.float32)
win_engine_ids = np.array(win_engine_ids)

print(f'windows: {X.shape},  labels: {y.shape}')

## 3. Train / validation split by engine

In [ ]:
# same split logic as notebook 03: avoids any engine's future cycles leaking into training
engine_ids = sorted(df['engine_id'].unique())
n_val      = max(1, int(len(engine_ids) * 0.2))
val_engines = set(engine_ids[-n_val:])

train_mask = ~np.isin(win_engine_ids, list(val_engines))
val_mask   =  np.isin(win_engine_ids, list(val_engines))

X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]

print(f'train: {X_train.shape}  val: {X_val.shape}')

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(WindowDataset(X_train, y_train), batch_size=256, shuffle=True)
val_loader   = DataLoader(WindowDataset(X_val,   y_val),   batch_size=512, shuffle=False)

## 4. Model definition

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.lstm1 = nn.LSTM(n_features, 64, batch_first=True)
        self.drop1 = nn.Dropout(0.2)
        self.lstm2 = nn.LSTM(64, 32, batch_first=True)
        self.drop2 = nn.Dropout(0.2)
        self.fc    = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.lstm1(x)
        out    = self.drop1(out)
        out, _ = self.lstm2(out)
        # only the last timestep carries the accumulated sequence context
        out    = self.drop2(out[:, -1, :])
        return self.fc(out).squeeze(1)


n_features = X_train.shape[2]
model      = LSTMRegressor(n_features).to(device)
print(model)

## 5. Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, factor=0.5, patience=5
)

MAX_EPOCHS     = 50
EARLY_STOP_PAT = 10

best_val_loss  = float('inf')
best_weights   = None
patience_count = 0
train_losses   = []
val_losses     = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        # 30-timestep sequences accumulate gradients; clipping prevents them from
        # blowing up the LSTM gates and locking the model into a constant output
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item() * len(yb)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_loss += criterion(model(xb), yb).item() * len(yb)
    val_loss /= len(val_loader.dataset)

    scheduler.step(val_loss)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f'epoch {epoch:02d}  train {train_loss:.4f}  val {val_loss:.4f}  lr {optimizer.param_groups[0]["lr"]:.6f}')

    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_weights   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= EARLY_STOP_PAT:
            print(f'early stop at epoch {epoch}')
            break

model.load_state_dict(best_weights)
print(f'best val loss: {best_val_loss:.4f}')

## 6. Evaluation

In [ ]:
def nasa_score(y_true, y_pred):
    d = np.array(y_pred) - np.array(y_true)
    # late predictions (positive d) penalised harder than early ones
    penalties = np.where(d < 0, np.expm1(-d / 13), np.expm1(d / 10))
    return penalties.sum()


model.eval()
all_preds = []
with torch.no_grad():
    for xb, _ in val_loader:
        all_preds.append(model(xb.to(device)).cpu().numpy())

preds  = np.concatenate(all_preds)
rmse   = np.sqrt(((preds - y_val) ** 2).mean())
score  = nasa_score(y_val, preds)

print(f'RMSE:       {rmse:.2f}')
print(f'NASA score: {score:.2f}  (lower is better)')

## 7. Predicted vs actual RUL

In [ ]:
val_engine_ids = win_engine_ids[val_mask]
val_cycles     = df.loc[df['engine_id'].isin(val_engines), 'cycle'].values

# reconstruct cycle index: each window label corresponds to its final cycle
val_final_cycles = []
for eid, group in df[df['engine_id'].isin(val_engines)].groupby('engine_id'):
    group = group.sort_values('cycle')
    for i in range(len(group) - WINDOW_SIZE + 1):
        val_final_cycles.append((eid, group['cycle'].iloc[i + WINDOW_SIZE - 1]))

plot_df = pd.DataFrame(val_final_cycles, columns=['engine_id', 'cycle'])
plot_df['rul']      = y_val
plot_df['pred_rul'] = preds

sample_engines = sorted(plot_df['engine_id'].unique())[:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=True)

for ax, eid in zip(axes.flat, sample_engines):
    eng = plot_df[plot_df['engine_id'] == eid].sort_values('cycle')
    ax.plot(eng['cycle'], eng['rul'],      label='actual',    lw=1.5)
    ax.plot(eng['cycle'], eng['pred_rul'], label='predicted', lw=1.5, linestyle='--')
    ax.set_title(f'engine {eid}')
    ax.set_xlabel('cycle')
    ax.set_ylabel('RUL')
    ax.legend(fontsize=8)

plt.suptitle('Predicted vs actual RUL, validation engines (LSTM)', y=1.01)
plt.tight_layout()
plt.show()

## 8. Loss curve

In [ ]:
epochs_ran = range(1, len(train_losses) + 1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs_ran, train_losses, label='train')
ax.plot(epochs_ran, val_losses,   label='val')
ax.set_xlabel('epoch')
ax.set_ylabel('MSE loss')
ax.set_title('Train vs validation loss')
ax.legend()
plt.tight_layout()
plt.show()